# LangGraph와 Llama Stack으로 에이전트 만들기

지난 노트북에서는 ReAct 에이전트를 **처음부터 직접** 만들었습니다. 다음과 같은 코드를 작성했습니다.
- 정규식을 이용한 수동 응답 파싱
- 직접 만든 반복 루프
- 상태 관리
- 도구 실행 로직
- 오류 처리

이제 **LangGraph**가 직접 구현한 실행 로직을 얼마나 간단하게 만들어 주는지 살펴보겠습니다.

## 만들게 될 것

다음 기능을 갖춘 지식 기반 어시스턴트입니다.
1. 문서에서 답을 검색합니다(RAG)
2. 답할 수 없으면 교수와의 미팅 일정을 잡습니다

유사한 Agentic 기능을 더 적은 실행 로직으로 구현합니다.

### 실습 목표
이전 노트북에서 직접 구현한 실행 루프를 프레임워크에 위임합니다. 여기서는 RAG와 캘린더에 더해 간단한 교수 디렉터리를 사용합니다. 두 실습의 도구 구성이 완전히 동일한 것은 아닙니다.

- **LangGraph:** 모델과 로컬 도구를 연결하는 실행 그래프, 반복 및 상태 관리.
- **Llama Stack:** 모델 추론, Vector Store API, Responses API 기반 원격 MCP 통합.
- **개발자:** 도구의 기능, 입력 규칙과 System Prompt에 정의할 업무 행동을 구성.



In [ ]:
# LangGraph 실행 그래프, 메시지 타입, 로컬 도구 데코레이터 및 Llama Stack 클라이언트를 준비합니다.
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from llama_stack_client import LlamaStackClient
import json

In [ ]:
# 라이브러리 로그를 줄여 실습 출력에 집중합니다.
import logging

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger().setLevel(logging.WARNING)

## Llama Stack에 연결하기

LangGraph는 MCP 도구 직접 바인딩 등 확장된 기능을 제공하는 **Responses API**를 사용해 Llama Stack에 연결합니다.

**중요**: `use_responses_api=True`를 사용할 때 base URL은 다음과 같아야 합니다.
- ✅ `http://llama-stack-service:8321/v1` (Responses API 엔드포인트)
- ❌ 사용하지 말 것: `http://llama-stack-service:8321/v1/openai/v1` (이것은 표준 Chat Completions용입니다)

매끄러운 MCP 통합을 가능하게 하는 것이 바로 Responses API입니다!

`ChatOpenAI`는 OpenAI 호환 인터페이스를 통해 Llama Stack에 요청하고 `use_responses_api=True`는 Responses API를 선택합니다. 별도 `LlamaStackClient`는 Vector Store 검색 등 RAG API에 사용됩니다. 이 셀은 클라이언트 구성 단계이며 서버가 실제로 응답하는지는 이후 호출에서 확인합니다.


In [ ]:
# LangGraph는 Llama Stack Responses API를 통해 연결합니다
# use_responses_api=True를 사용할 때 base는 /v1이어야 합니다(/v1/openai/v1이 아님)
# Responses API는 MCP 통합을 위한 도구 바인딩을 지원합니다
llm = ChatOpenAI(
    openai_api_base="http://llama-stack-service:8321/v1",
    model="vllm-llama32/llama32",
    openai_api_key="not-needed",
    use_responses_api=True,
    temperature=0.1
)

# RAG용 표준 클라이언트도 연결
llama_client = LlamaStackClient(base_url="http://llama-stack-service:8321")

print(f"✅ Responses API가 활성화된 Llama Stack에 연결 완료")

## 도구 정의하기

에이전트가 다음 도구를 사용할 수 있게 합니다.
1. **RAG 검색** - 지식 베이스에서 정보 검색
2. **교수 디렉터리** - 전문 분야로 교수 찾기  
3. **MCP 캘린더 도구** - MCP 서버 직접 통합!

MCP 캘린더 통합에는 **직접 도구 바인딩**을 지원하는 Llama Stack의 Responses API를 사용합니다. 도구 목록에 MCP 바인딩을 넣으면 LangGraph와 Llama Stack이 실행 과정의 많은 부분을 처리합니다.

- MCP 바인딩은 앞에서 설정한 실제 캘린더 서버에 연결됩니다
- 9개의 캘린더 도구(create_event, get_upcoming_events, search_events 등)를 모두 사용할 수 있게 됩니다
- 원격 MCP 도구 호출은 Llama Stack이 처리합니다
- 래퍼 함수를 직접 만들 필요가 없습니다!

이전 노트북(`2-mcp-servers.ipynb`, `3-agentic-workflows.ipynb`)에서 사용한 것과 같은 MCP 캘린더 서버입니다.


In [ ]:
import io
import time

store_name = "canopy_agentic_rag"
embedding_model_id = "sentence-transformers/nomic-ai/nomic-embed-text-v1.5"
embedding_dimension = 768
TEST_QUERY = "What is the structure of Canopy in botany?"
vector_store = None


def create_lab_store():
    store = llama_client.vector_stores.create(
        name=store_name,
        extra_body={
            "embedding_model": embedding_model_id,
            "embedding_dimension": embedding_dimension,
            "provider_id": "milvus",
            "vector_db_id": store_name
        }
    )
    if store.id != store_name:
        raise RuntimeError(f"예상과 다른 저장소 ID: {store.id}")
    return store


def test_search(store):
    return llama_client.vector_stores.search(
        vector_store_id=store.id,
        query=TEST_QUERY,
        max_num_results=5,
        search_mode="vector"
    )


# 1. 실습용 저장소 조회 또는 생성
selected_store = next(
    (vs for vs in llama_client.vector_stores.list().data if vs.id == store_name),
    None
)
if selected_store is None:
    selected_store = create_lab_store()
    print("✅ 실습용 벡터 저장소 생성 완료")

# 2. 등록 문서의 상태 및 실제 검색 가능 여부 확인
existing_files = llama_client.vector_stores.files.list(vector_store_id=selected_store.id).data
has_completed = any(f.status == "completed" for f in existing_files)
has_in_progress = any(f.status == "in_progress" for f in existing_files)
search_results = test_search(selected_store) if has_completed else None

if has_completed and not search_results.data:
    # 검색 API 예외는 그대로 전파하며, 결과가 비어 있을 때만 재생성합니다.
    if has_in_progress or selected_store.id != store_name or selected_store.name != store_name:
        raise RuntimeError("인덱싱 진행 중이거나 실습용 저장소가 아니므로 삭제할 수 없습니다.")
    print("⚠️ 검색 결과 없음: 실습 전용 저장소 재생성")
    llama_client.vector_stores.delete(vector_store_id=selected_store.id)
    selected_store = create_lab_store()
    has_completed = False

# 3. 정상적으로 검색 가능한 문서가 없으면 샘플 문서를 등록
if not has_completed:
    if has_in_progress:
        raise RuntimeError("인덱싱 진행 중입니다. 완료 후 다시 실행하세요.")

    document = """
Canopy in Botany

In botany, a canopy is the aboveground portion of a plant
community formed by the crowns of trees and other vegetation.

Structure of a Forest Canopy

The vertical structure of a tropical rainforest typically
consists of four layers:

1. Emergent Layer:
   The tallest trees extending above the main forest canopy.

2. Canopy Layer:
   A dense layer of overlapping tree crowns that intercepts
   most of the sunlight.

3. Understory Layer:
   Smaller trees, shrubs, and plants adapted to lower
   light conditions.

4. Forest Floor:
   The lowest layer, containing decomposing organic matter
   and shade-tolerant vegetation.

Functions of the Canopy

The canopy regulates temperature, humidity, sunlight,
and rainfall interception. It provides habitats for
numerous species and plays an important role in
photosynthesis and carbon storage.
"""
    uploaded_file = llama_client.files.create(
        file=("canopy_botany.txt", io.BytesIO(document.encode("utf-8"))),
        purpose="assistants"
    )
    indexed_file = llama_client.vector_stores.files.create(
        vector_store_id=selected_store.id,
        file_id=uploaded_file.id,
        chunking_strategy={
            "type": "static",
            "static": {"max_chunk_size_tokens": 512, "chunk_overlap_tokens": 64}
        }
    )

    # 최대 120초 동안 인덱싱 완료 대기 (API 호출 시간 제외)
    for _ in range(60):
        if indexed_file.status != "in_progress":
            break
        time.sleep(2)
        indexed_file = llama_client.vector_stores.files.retrieve(
            vector_store_id=selected_store.id, file_id=uploaded_file.id
        )
    if indexed_file.status == "in_progress":
        raise TimeoutError("문서 인덱싱 완료 대기 시간 초과")
    if indexed_file.status != "completed":
        raise RuntimeError(f"문서 인덱싱 실패: {indexed_file}")

    print("✅ 문서 임베딩 및 인덱싱 완료")
    search_results = test_search(selected_store)
else:
    print(f"✅ 기존 Vector Store 및 문서 재사용: {selected_store.id}")

# 4. 검색 성공 후에만 RAG 활성화
if not search_results.data:
    raise RuntimeError("벡터 저장소에서 관련 문서를 검색하지 못했습니다.")

vector_store = selected_store
print(f"✅ RAG 시스템 초기화 완료: {vector_store.id}")
print(f"검색 결과: {len(search_results.data)}개")

위의 코드를 실행했을 때 다른 이유 설명 없이 '❌ 벡터 저장소 설정 실패' 문구만 뜬다면 잠시 기다렸다가 다시 실행해보세요. 

In [ ]:
# 지금 보시는 교수 데이터는 실습용 딕셔너리입니다. 실제 시스템에서는 데이터베이스를 사용할 것입니다. 
# @tool은 로컬 도구를 등록하고 MCP 설정 객체는 원격 도구를 선언합니다.
PROFESSORS = {
    "Dr. Sarah Chen": {
        "department": "Computer Science",
        "expertise": ["Machine Learning", "Neural Networks", "AI Ethics", "Agentic Workflows"],
        "email": "s.chen@university.edu"
    },
    "Prof. Michael Rodriguez": {
        "department": "Physics",
        "expertise": ["Quantum Mechanics", "Particle Physics", "Quantum Chromodynamics"],
        "email": "m.rodriguez@university.edu"
    },
    "Dr. Emily Thompson": {
        "department": "Biology",
        "expertise": ["Botany", "Ecology", "Forest Canopy Structure", "Plant Biology"],
        "email": "e.thompson@university.edu"
    },
    "Prof. James Wilson": {
        "department": "Computer Science",
        "expertise": ["Distributed Systems", "Cloud Computing", "Software Architecture"],
        "email": "j.wilson@university.edu"
    }
}


@tool
def search_knowledge_base(query: str) -> str:
    """Search through documents to find information. Use this when the user asks about concepts, definitions, or topics."""
    if not vector_store:
        return "Error: Knowledge base not available. Please set up the vector store first."
    
    try:
        results = llama_client.vector_stores.search(
            vector_store_id=vector_store.id,
            query=query,
            max_num_results=3,
            search_mode="vector"
        )
        
        if not results.data:
            return "No relevant information found in the knowledge base."
        
        formatted_results = []
        for i, result in enumerate(results.data, 1):
            content = result.content if hasattr(result, 'content') else str(result)
            formatted_results.append(f"Result {i}: {content}")
        
        return "\n\n".join(formatted_results)
    except Exception as e:
        return f"Error searching knowledge base: {str(e)}"


@tool
def find_professors_by_expertise(topic: str) -> str:
    """Find professors who have expertise in a specific topic or subject area.
    
    Args:
        topic: The topic or subject area to search for (e.g., 'Machine Learning', 'Quantum Physics', 'Botany')
    
    Returns:
        List of professors with matching expertise, including their name, department, and contact info
    """
    matching_profs = []
    
    for name, info in PROFESSORS.items():
        # 주제가 교수의 전문 분야 중 하나와 일치하는지 확인
        if any(topic.lower() in exp.lower() or exp.lower() in topic.lower() for exp in info["expertise"]):
            matching_profs.append((name, info))
    
    if not matching_profs:
        # 정확히 일치하는 교수가 없으면 모든 교수를 표시
        result = f"No professors found with specific expertise in '{topic}'.\n\n"
        result += "Available professors:\n\n"
        for name, info in PROFESSORS.items():
            result += f"**{name}** - {info['department']}\n"
            result += f"  Expertise: {', '.join(info['expertise'])}\n"
            result += f"  Email: {info['email']}\n\n"
        return result
    
    result = f"Professors with expertise in '{topic}':\n\n"
    for name, info in matching_profs:
        result += f"**{name}** - {info['department']}\n"
        result += f"  Expertise: {', '.join(info['expertise'])}\n"
        result += f"  Email: {info['email']}\n\n"
    
    return result


# 전체 도구 정의: 커스텀 도구 + MCP 바인딩
tools = [
    search_knowledge_base,
    find_professors_by_expertise,
    # MCP 캘린더 서버 바인딩
    {
        "type": "mcp",
        "server_label": "canopy-calendar",
        "server_url": "http://canopy-mcp-calendar-mcp-server:8080/sse",
        "require_approval": "never",
    }
]

print("✅ 정의된 도구:")
print("   사용자 정의 도구:")
print("     - search_knowledge_base")
print("     - find_professors_by_expertise")
print("   MCP 일정 관리 서버:")
print("     - canopy-calendar (Responses API를 통해 일정 관리 도구 9개 제공)")

## 에이전트 만들기

LangGraph를 사용하면 ReAct 에이전트를 **함수 호출 하나**로 구성할 수 있습니다.

수동 텍스트 파싱, 기본 실행 루프와 상태 관리 로직을 직접 작성할 필요가 없습니다.

**내부에서 일어나는 일:**
- `tools` 목록에 MCP 캘린더 바인딩을 전달합니다
- `create_react_agent()`가 모든 도구(커스텀 + MCP)를 LLM에 자동으로 바인딩합니다
- Responses API가 MCP 도구 바인딩을 지원하므로 MCP 서버와 직접 통합할 수 있습니다
- 원격 MCP 호출은 Llama Stack의 Responses API가 처리하고 LangGraph는 전체 에이전트 실행 흐름을 관리합니다
- 이제 에이전트가 실제 캘린더 데이터베이스와 상호작용할 수 있습니다!


In [ ]:

from datetime import datetime
# 업무 행동 규칙은 아래와 같이 System Prompt에 두고 실행 루프는 프레임워크에 위임합니다.
system_prompt = f"""You are a helpful assistant that helps students
with their calendar and studies.

Today is {datetime.today().strftime('%Y-%m-%d')}.

## Calendar Questions
- Use the appropriate MCP calendar tools to retrieve schedules.
- Filter by category only when requested.
- Do not modify events unless explicitly requested.

## Knowledge Questions
- Use search_knowledge_base to find relevant information.
- If relevant information is found, answer using the results.
- If no relevant information is found, recommend an expert.

## Expert Search and Meeting Scheduling
When the student requests an expert or a meeting:
1. Use find_professors_by_expertise to find a relevant professor.
2. If a meeting is requested, check that the date and time are valid and in the future.
3. Use get_events_by_date to check for scheduling conflicts.
4. If the requested time is available, call create_event.
5. Confirm the meeting only after successful event creation.

## Event Creation Rules
Use the following parameters for create_event:
- name: Meeting title including the professor's name.
- category: "Meeting"
- level: 2 (default priority).
- start_time and end_time: YYYY-MM-DD HH:MM:SS.
- content: Professor's name and meeting topic.

Do not include sid, status, or creation_time.
Avoid scheduling conflicts and duplicate events.

## General Rules
- Follow each tool's input schema strictly.
- Execute required tools instead of describing intended actions.
- Complete all requested tasks before returning a final answer.
- Never fabricate tool results or claim success without confirmation.
- If a tool fails, report the error or retry safely.
"""

In [ ]:
# 모든 도구를 사용하는 ReAct 에이전트 생성. 이전에 비해 에이전트 코드가 굉장히 단순해졌습니다. 
# 사용할 모델, 도구, 프롬프트 및 체크포인터를 다음과 같이 함수에 전달하면 에이전트를 쉽게 구성합니다.
agent = create_react_agent(
    llm,
    tools,
    prompt=system_prompt,
    checkpointer=MemorySaver(),
)

print("✅ ReAct 에이전트 생성 완료!")
print("\n🎯 에이전트가 사용할 수 있는 도구:")
print("   • 사용자 정의 도구 2개 (RAG 검색, 교수진 디렉터리)")
print("   • MCP 일정 관리 도구 9개 (Responses API를 통해 자동 연결)")
print("\n🎯 앞서 작성한 200줄 이상의 코드와 비교해 보세요...")

에이전트를 간단한 다이어그램으로도 볼 수 있습니다.

아래 셀의 객체 표시 결과는 실행 환경과 라이브러리 버전에 따라 달라질 수 있습니다. 그래프가 표시된다면 모델 호출과 도구 실행이 연결된 구조를 확인하세요.


In [ ]:
agent

### 방금 무슨 일이 일어났나요?

함수 호출 하나로 이전 노트북의 다음 코드가 모두 대체되었습니다.
- ❌ `parse_react_response()` - 정규식을 이용한 수동 파싱
- ❌ `execute_tool()` - 도구 라우팅 로직
- ❌ `run_react_agent()` - 전체 반복 루프
- ❌ 대화 기록 관리
- ❌ 기본 반복 및 종료 조건 검사

LangGraph가 기본 실행 로직을 제공하므로 개발자는 모델과 도구 및 행동 규칙에 집중할 수 있습니다.

프레임워크가 제공하는 기본 실행 루프와 구조화된 도구 호출 처리 덕분에 이전의 수동 정규식 파서와 라우터를 직접 작성할 필요가 없습니다. 그러나 모든 업무 오류에 대한 재시도나 실제 예약 성공 검증까지 자동으로 보장하지는 않습니다. 운영 환경에서는 작업 완료 조건, 입력 검증 및 외부 시스템 결과 확인이 별도로 필요할 수 있습니다.


## 에이전트 테스트하기

실제로 동작하는 모습을 확인해 봅시다!

세 예제에서는 문서 검색 → 전문가 검색 및 일정 등록 → 캘린더 재조회 순서로 읽기와 쓰기 기능을 확인합니다. 최종 응답만 읽지 말고 도구 실행 로그와 실제 일정 데이터도 확인하세요.

먼저 에이전트의 출력을 보여 주는 로직을 추가한 뒤, 이어지는 예제에서 에이전트를 실행해 어떤 결과가 나오는지 확인합니다.


In [ ]:
# 새로운 그래프 상태를 읽어 도구 호출과 최종 응답을 출력합니다. 동일 thread_id는 상태 재사용에 활용됩니다.
import uuid
from textwrap import indent
from langchain_core.messages import HumanMessage
import pprint

def run_agent(question: str, thread_id: str | None = None):
    """LangGraph 에이전트를 실행하고 값을 스트리밍하며, MCP 도구 호출과 최종 답변을 표시합니다."""
    thread_id = thread_id or str(uuid.uuid4())

    print("\n" + "="*80)
    print("사용자 입력")
    print("="*80 + "\n")
    print(question, "\n")

    config = {"configurable": {"thread_id": thread_id}}
    inputs = {"messages": [HumanMessage(content=question)]}

    seen_messages = 0
    final_state = None
    tool_header_printed = False

    # 중요: stream_mode="values"
    for state in agent.stream(inputs, config, stream_mode="values"):

        messages = state.get("messages", [])
        new_messages = messages[seen_messages:]
        seen_messages = len(messages)

        for msg in new_messages:
            msg_type = getattr(msg, "type", None)

            if msg_type == "ai":
                if getattr(msg, "tool_calls", None):
                    if not tool_header_printed:
                        print("="*80)
                        print("도구 호출")
                        print("="*80 + "\n")
                        tool_header_printed = True

                    for tc in msg.tool_calls:
                        name = tc.get("name")
                        args = tc.get("args")
                        print(f"🔧 도구 호출: {name}")
                        print(f"   인수: {args}\n")
                else:
                    tool_outputs = msg.additional_kwargs.get("tool_outputs", [])

                    # MCP 도구 호출 정보는 여기에 있음
                    for t in tool_outputs:
                        if t.get("type") == "mcp_call":
                            if not tool_header_printed:
                                print("="*80)
                                print("도구 호출")
                                print("="*80 + "\n")
                                tool_header_printed = True

                            name = t.get("name")
                            server = t.get("server_label")
                            args = t.get("arguments")
                            error = t.get("error", "")
                            output = t.get("output", "")

                            print(f"🔧 MCP 도구 호출: {name}  (서버: {server})")
                            print(f"   인수: {args}")
                            if output:
                                print("   출력:")
                                print(indent(str(output), "      "))
                            elif error:
                                print("   오류:")
                                print(indent(str(error), "      "))
                            print()

            elif msg_type == "tool":
                # 결과를 얼마나 보여 줄지 원하는 대로 조정할 수 있음
                print(f"📦 도구 실행 결과 ({msg.name}): {str(msg.content)[:200]}...\n")

        final_state = state

    if final_state is None:
        print("⚠️ 에이전트가 상태를 생성하지 않았습니다")
        return

    # 최종 어시스턴트 메시지 = 마지막 AI 메시지
    messages = final_state.get("messages", [])
    final_answer = None
    for msg in reversed(messages):
        if getattr(msg, "type", None) == "ai":
            final_answer = msg
            break

    print("="*80)
    print("에이전트의 최종 응답")
    print("="*80 + "\n")
    if final_answer:
        content = final_answer.content
        if isinstance(content, list):
            for item in content:
                if isinstance(item, dict) and "text" in item:
                    print(item["text"])
                elif hasattr(item, "text"):
                    print(item.text)
        else:
            print(content)
    else:
        print("(어시스턴트 응답을 찾을 수 없습니다)")
    print("="*80 + "\n")


### 예제 1: 지식 베이스 검색

에이전트는 이 정보를 지식 베이스에서 검색해야 합니다.

**참고**: `3-agentic-workflows.ipynb`의 도구 3에서 생성한 `canopy_agentic_rag` 저장소와 정상적으로 인덱싱된 문서를 재사용합니다.

예상 동작은 `search_knowledge_base` 호출 후 식물학 Canopy 관련 문서를 근거로 답변하는 것입니다. 3번 노트북에서 생성한 저장소와 문서가 같은 서버에 있어야 합니다. 초기화가 실패하면 3번의 도구 3 결과부터 확인하세요.


In [ ]:
run_agent("What is the structure of a forest canopy in botany?")

### 예제 2: 전문가 찾기와 일정 잡기

지식 검색으로 충분한 답을 얻지 못하고 사용자가 전문가와의 미팅을 요청했을 때 다음을 수행하는지 지켜보세요.
1. 관련 전문 분야의 교수 검색
2. 캘린더에 미팅 일정 등록

실습용 교수 디렉터리에는 양자 색역학(Quantum Chromodynamics)을 전문 분야로 하는 Prof. Michael Rodriguez가 등록되어 있습니다. Canopy 문서가 반환되더라도 질문과 관련 없는 자료로 판단해야 합니다.

기대 흐름: 지식 검색 → 전문가 검색 → 해당 날짜 일정 조회 → 가능한 경우 `create_event` → 실제 생성 결과 확인. 코드에서는 실행일 기준 내일 16시부터 1시간을 요청합니다. 기존 일정과 겹치거나 동일한 실습을 반복했다면 예약이 실패하거나 중복될 수 있습니다. 학생 캘린더에 이벤트를 등록하는 것은 실제 교수의 참석 확인이나 초대 발송을 의미하지 않습니다.


In [ ]:
# 실행일 기준 내일 날짜를 계산합니다. 실제 이벤트 생성이 포함될 수 있습니다.
from datetime import datetime, timedelta

# 내일 날짜를 미팅 요청 날짜로 설정
meeting_date = (
    datetime.today() + timedelta(days=1)
).strftime("%Y-%m-%d")

run_agent(
    "I need help understanding quantum chromodynamics. "
    "Can you find me an expert and schedule a meeting "
    f"for {meeting_date} at 4pm for one hour?"
)

### 예제 3: MCP 캘린더 사용하기

에이전트가 실제 캘린더를 확인하고 일정을 검색할 수 있습니다! 미팅 목록이 전부 안 보인다면 출력 결과에서 'scrollable element'를 클릭해보세요.

예제 2에서 생성한 미팅을 포함해 앞으로 7일간의 전체 일정을 조회합니다. 

In [ ]:
# 전체 카테고리의 예정 일정을 조회해 앞서 등록한 미팅도 확인합니다.
run_agent(
    "What upcoming events do I have in the next 7 days? "
    "Include all categories, especially meetings and lectures."
)

## 직접 해 보세요!

이제 실험해 보세요.
1. 지식 베이스 검색이 필요한 질문하기
2. 미팅 일정 등록이 필요한 질문하기
3. 에이전트가 두 도구를 차례로 사용하도록 만들어 보기

에이전트가 어떤 도구를 사용할지 스스로 추론하는 과정을 확인해 보세요!

**다음 실습:** 지금까지 인증 없이 접근한 MCP 서버를 OAuth 액세스 토큰으로 보호하는 방법을 살펴봅니다.


In [ ]:
# 여러분 차례입니다!
run_agent("YOUR QUESTION HERE")